# Neural Quantum States – Parameter Study (Poster)

Explores how VMC accuracy depends on model parameters in three exactly solvable 1D systems:
1. **Harmonic oscillator** – N independent particles
2. **Open chain** – nearest-neighbour harmonic coupling (OBC)
3. **Periodic chain** – nearest-neighbour harmonic coupling (PBC)

All models are **log-wavefunctions**. Architectures tested:
- `mlp-fourth-decay` (log MLP with x⁴ envelope)
- `mlp-gaussian-decay` (log MLP with x² envelope)
- `deep-set` (log DeepSet with x⁴ envelope)

A `USE_JACOBI` flag (default `False`) controls whether the chain is sampled in
Jacobi relative coordinates (removes centre-of-mass degree of freedom).

In [ ]:
import sys
sys.path.insert(0, '../../../src')

import jax
import jax.numpy as jnp
import numpy as np
import optax
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from itertools import product
from tqdm.notebook import tqdm

from flax import linen as nn

from qvarnet.train import train
from qvarnet.config.training_setup import TrainingConfig, parse_sampler_params
from qvarnet.config.coord_mode import LabCoords, JacobiCoords
from qvarnet.hamiltonian.continuous import HarmonicOscillatorHamiltonian, NN_OscillatorHamiltonian
from qvarnet.models.exponential import LogExponentialMLPwithPenalty, LogExponentialMLPwithGaussianPenalty
from qvarnet.models.deep_set import DeepSet

## Global configuration

In [ ]:
# ── System ────────────────────────────────────────────────────────────────────
N_PARTICLES  = 5          # number of particles
DIM          = 1          # spatial dimension
DoF          = N_PARTICLES * DIM

OMEGA_TRAP   = 1.0        # trapping frequency
OMEGA_INT    = 1.0        # nearest-neighbour coupling

# ── Jacobi coordinates flag ───────────────────────────────────────────────────
# When True the sampler works in N-1 Jacobi relative coords; the model still
# receives N lab coords reconstructed from them. Default: False.
USE_JACOBI   = False

# ── Architecture search space ─────────────────────────────────────────────────
HIDDEN_WIDTHS  = [16, 32, 64]   # number of hidden units per layer
HIDDEN_DEPTHS  = [1, 2, 3]      # number of hidden layers

# ── Training ──────────────────────────────────────────────────────────────────
N_EPOCHS             = 3_000
N_CHAINS             = 500
RNG_SEED             = 42
LEARNING_RATE        = 1e-3
WARM_WALKERS         = True
IS_UPDATE_STEP_SIZE  = True
TARGET_ACCEPTANCE    = 0.5

# ── Sampler ───────────────────────────────────────────────────────────────────
sampler_params = {
    "step_size":            0.5,
    "chain_length":         20,
    "thermalization_steps": 50,
    "thinning_factor":      2,
    "PBC":                  40.0,
}

# ── Shape ─────────────────────────────────────────────────────────────────────
# For Jacobi: sampler shape has N-1 DoF; model shape has N DoF
if USE_JACOBI:
    SHAPE = (N_CHAINS, DoF - DIM)   # sampler works in N-1 Jacobi coords
else:
    SHAPE = (N_CHAINS, DoF)

print(f"N_PARTICLES={N_PARTICLES}, DoF={DoF}, USE_JACOBI={USE_JACOBI}")
print(f"Sampler shape: {SHAPE}")

## Exact ground-state energies

In [ ]:
def exact_energy_ho(n_particles, omega=1.0):
    """E₀ = N * omega/2  (N independent 1D harmonic oscillators, ℏ=m=1)."""
    return n_particles * omega / 2


def _coupling_matrix_obc(n, omega_trap, omega_int):
    """Potential coupling matrix for the OBC nearest-neighbour chain."""
    K = np.zeros((n, n))
    # endpoints appear in exactly one interaction term
    K[0, 0] = omega_trap**2 + omega_int**2
    K[n-1, n-1] = omega_trap**2 + omega_int**2
    for i in range(1, n - 1):
        K[i, i] = omega_trap**2 + 2 * omega_int**2
    for i in range(n - 1):
        K[i, i+1] = -omega_int**2
        K[i+1, i] = -omega_int**2
    return K


def _coupling_matrix_pbc(n, omega_trap, omega_int):
    """Potential coupling matrix for the PBC nearest-neighbour chain."""
    K = np.zeros((n, n))
    for i in range(n):
        K[i, i] = omega_trap**2 + 2 * omega_int**2
    for i in range(n):
        K[i, (i+1) % n] = -omega_int**2
        K[(i+1) % n, i] = -omega_int**2
    return K


def exact_energy_chain_obc(n_particles, omega_trap=1.0, omega_int=1.0):
    """E₀ = (1/2) * sum of normal-mode frequencies for OBC chain."""
    K = _coupling_matrix_obc(n_particles, omega_trap, omega_int)
    eigvals = np.linalg.eigvalsh(K)
    return 0.5 * np.sum(np.sqrt(np.maximum(eigvals, 0.0)))


def exact_energy_chain_pbc(n_particles, omega_trap=1.0, omega_int=1.0):
    """E₀ = (1/2) * sum of normal-mode frequencies for PBC chain."""
    K = _coupling_matrix_pbc(n_particles, omega_trap, omega_int)
    eigvals = np.linalg.eigvalsh(K)
    return 0.5 * np.sum(np.sqrt(np.maximum(eigvals, 0.0)))


E_HO  = exact_energy_ho(N_PARTICLES, OMEGA_TRAP)
E_OBC = exact_energy_chain_obc(N_PARTICLES, OMEGA_TRAP, OMEGA_INT)
E_PBC = exact_energy_chain_pbc(N_PARTICLES, OMEGA_TRAP, OMEGA_INT)

print(f"Exact E₀ (HO)        : {E_HO:.6f}")
print(f"Exact E₀ (chain OBC) : {E_OBC:.6f}")
print(f"Exact E₀ (chain PBC) : {E_PBC:.6f}")

## Model builders

In [ ]:
def build_mlp_fourth_decay(n_dof, width, depth):
    """Log MLP with x⁴ decay envelope.  arch = [n_dof, width×depth, 1]."""
    architecture = [n_dof] + [width] * depth + [1]
    return LogExponentialMLPwithPenalty(
        architecture=architecture,
        hidden_activation=nn.tanh,
        kernel_init=nn.initializers.lecun_normal(),
        bias_init=nn.initializers.zeros_init(),
    )


def build_mlp_gaussian_decay(n_dof, width, depth):
    """Log MLP with x² (Gaussian) decay envelope.  arch = [n_dof, width×depth, 1]."""
    architecture = [n_dof] + [width] * depth + [1]
    return LogExponentialMLPwithGaussianPenalty(
        architecture=architecture,
        hidden_activation=nn.tanh,
    )


def build_deepset(n_particles, width, depth, hidden_internal_dim=None):
    """Log DeepSet.  phi_arch = F_arch = [width×depth]; hidden_internal_dim = width."""
    if hidden_internal_dim is None:
        hidden_internal_dim = width
    phi_hidden = [width] * depth
    F_hidden   = [width] * depth
    return DeepSet(
        n_particles=n_particles,
        n_dim=DIM,
        phi_hidden_architecture=phi_hidden,
        F_hidden_architecture=F_hidden,
        hidden_internal_dimension=hidden_internal_dim,
        phi_hidden_activation=nn.tanh,
        F_hidden_activation=nn.tanh,
        kernel_init=nn.initializers.lecun_normal(),
        bias_init=nn.initializers.zeros_init(),
    )


def n_params(model, shape):
    """Count total trainable parameters."""
    key = jax.random.PRNGKey(0)
    params = model.init(key, jnp.ones(shape[1:]))
    return sum(x.size for x in jax.tree_util.tree_leaves(params))

## Training helper

In [ ]:
def run_vmc(model, hamiltonian, shape, n_epochs=N_EPOCHS, rng_seed=RNG_SEED,
            learning_rate=LEARNING_RATE, use_jacobi=USE_JACOBI):
    """Train a log-wavefunction model via VMC and return the history."""
    optimizer = optax.adam(learning_rate=learning_rate)

    training_config = TrainingConfig(
        n_epochs=n_epochs,
        rng_seed=rng_seed,
        is_log_model=True,          # all models are log-wavefunctions
        warm_walkers=WARM_WALKERS,
        is_update_step_size=IS_UPDATE_STEP_SIZE,
        min_step=1e-5,
        max_step=5.0,
        target_acceptance=TARGET_ACCEPTANCE,
        save_checkpoints=False,
    )

    if use_jacobi:
        coord_mode = JacobiCoords(n_particles_physical=N_PARTICLES, n_dim=DIM)
    else:
        coord_mode = LabCoords()

    result = train(
        shape=shape,
        model=model,
        optimizer=optimizer,
        hamiltonian=hamiltonian,
        training_config=training_config,
        sampler_params=sampler_params,
        coord_mode=coord_mode,
    )
    return result


def extract_energies(result):
    """Return energies array from a TrainResult."""
    return np.array([float(s.energy) for s in result.history])


def tail_energy(energies, tail=200):
    """Mean energy over the last `tail` epochs."""
    return float(np.mean(energies[-tail:]))

## System definitions

In [ ]:
hamiltonian_ho  = HarmonicOscillatorHamiltonian(omega=OMEGA_TRAP)
hamiltonian_obc = NN_OscillatorHamiltonian(omega_trap=OMEGA_TRAP, omega_interaction=OMEGA_INT, with_pbc=False)
hamiltonian_pbc = NN_OscillatorHamiltonian(omega_trap=OMEGA_TRAP, omega_interaction=OMEGA_INT, with_pbc=True)

SYSTEMS = {
    "HO":        (hamiltonian_ho,  E_HO),
    "chain_obc": (hamiltonian_obc, E_OBC),
    "chain_pbc": (hamiltonian_pbc, E_PBC),
}

print("Systems defined:")
for name, (_, E) in SYSTEMS.items():
    print(f"  {name:12s}  E₀ = {E:.6f}")

## Architecture × system experiment grid

For each `(system, architecture_type, width, depth)` combination we train one model
and record the final variational energy.

In [ ]:
ARCH_TYPES = ["mlp_fourth_decay", "mlp_gaussian_decay", "deep_set"]

# results[(system_name, arch_type, width, depth)] = {
#   'energies': np.array, 'final_E': float, 'error': float, 'n_params': int
# }
results = {}


def build_model(arch_type, width, depth):
    if arch_type == "mlp_fourth_decay":
        return build_mlp_fourth_decay(DoF, width, depth)
    elif arch_type == "mlp_gaussian_decay":
        return build_mlp_gaussian_decay(DoF, width, depth)
    elif arch_type == "deep_set":
        return build_deepset(N_PARTICLES, width, depth)
    else:
        raise ValueError(f"Unknown arch_type: {arch_type}")


total_runs = len(SYSTEMS) * len(ARCH_TYPES) * len(HIDDEN_WIDTHS) * len(HIDDEN_DEPTHS)
print(f"Total runs: {total_runs}")

### Harmonic oscillator

In [ ]:
system_name = "HO"
hamiltonian, E_exact = SYSTEMS[system_name]

for arch_type, width, depth in tqdm(
    list(product(ARCH_TYPES, HIDDEN_WIDTHS, HIDDEN_DEPTHS)),
    desc=system_name,
):
    key = (system_name, arch_type, width, depth)
    model = build_model(arch_type, width, depth)
    n_par = n_params(model, SHAPE)

    result    = run_vmc(model, hamiltonian, SHAPE)
    energies  = extract_energies(result)
    final_E   = tail_energy(energies)

    results[key] = {
        "energies": energies,
        "final_E":  final_E,
        "error":    abs(final_E - E_exact),
        "rel_error": abs(final_E - E_exact) / abs(E_exact),
        "n_params": n_par,
    }
    print(f"  {arch_type:22s}  w={width:3d}  d={depth}  "
          f"n_par={n_par:6d}  E={final_E:.4f}  err={results[key]['error']:.4f}")

### Open chain (OBC)

In [ ]:
system_name = "chain_obc"
hamiltonian, E_exact = SYSTEMS[system_name]

for arch_type, width, depth in tqdm(
    list(product(ARCH_TYPES, HIDDEN_WIDTHS, HIDDEN_DEPTHS)),
    desc=system_name,
):
    key = (system_name, arch_type, width, depth)
    model = build_model(arch_type, width, depth)
    n_par = n_params(model, SHAPE)

    result    = run_vmc(model, hamiltonian, SHAPE)
    energies  = extract_energies(result)
    final_E   = tail_energy(energies)

    results[key] = {
        "energies": energies,
        "final_E":  final_E,
        "error":    abs(final_E - E_exact),
        "rel_error": abs(final_E - E_exact) / abs(E_exact),
        "n_params": n_par,
    }
    print(f"  {arch_type:22s}  w={width:3d}  d={depth}  "
          f"n_par={n_par:6d}  E={final_E:.4f}  err={results[key]['error']:.4f}")

### Periodic chain (PBC)

In [ ]:
system_name = "chain_pbc"
hamiltonian, E_exact = SYSTEMS[system_name]

for arch_type, width, depth in tqdm(
    list(product(ARCH_TYPES, HIDDEN_WIDTHS, HIDDEN_DEPTHS)),
    desc=system_name,
):
    key = (system_name, arch_type, width, depth)
    model = build_model(arch_type, width, depth)
    n_par = n_params(model, SHAPE)

    result    = run_vmc(model, hamiltonian, SHAPE)
    energies  = extract_energies(result)
    final_E   = tail_energy(energies)

    results[key] = {
        "energies": energies,
        "final_E":  final_E,
        "error":    abs(final_E - E_exact),
        "rel_error": abs(final_E - E_exact) / abs(E_exact),
        "n_params": n_par,
    }
    print(f"  {arch_type:22s}  w={width:3d}  d={depth}  "
          f"n_par={n_par:6d}  E={final_E:.4f}  err={results[key]['error']:.4f}")

## Results summary

In [ ]:
import pandas as pd

rows = []
for (sys_name, arch_type, width, depth), v in results.items():
    rows.append({
        "system":    sys_name,
        "arch":      arch_type,
        "width":     width,
        "depth":     depth,
        "n_params":  v["n_params"],
        "final_E":   v["final_E"],
        "error":     v["error"],
        "rel_error": v["rel_error"],
    })

df = pd.DataFrame(rows)
df = df.sort_values(["system", "arch", "width", "depth"]).reset_index(drop=True)
print(df.to_string(index=False))

## Visualisation

### Error vs number of parameters (log–log)

In [ ]:
ARCH_COLORS  = {"mlp_fourth_decay": "C0", "mlp_gaussian_decay": "C1", "deep_set": "C2"}
ARCH_MARKERS = {"mlp_fourth_decay": "o",  "mlp_gaussian_decay": "s",  "deep_set": "^"}
ARCH_LABELS  = {"mlp_fourth_decay": "MLP (x⁴ decay)",
                "mlp_gaussian_decay": "MLP (Gaussian decay)",
                "deep_set": "DeepSet"}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, sys_name in zip(axes, ["HO", "chain_obc", "chain_pbc"]):
    sub = df[df["system"] == sys_name]
    for arch_type in ARCH_TYPES:
        sub_a = sub[sub["arch"] == arch_type].sort_values("n_params")
        ax.loglog(
            sub_a["n_params"], sub_a["rel_error"],
            marker=ARCH_MARKERS[arch_type],
            color=ARCH_COLORS[arch_type],
            label=ARCH_LABELS[arch_type],
            linewidth=1.5, markersize=7, alpha=0.85,
        )
    ax.set_xlabel("Number of parameters", fontsize=12)
    ax.set_ylabel("Relative energy error  |E−E₀|/E₀", fontsize=11)
    ax.set_title({"HO": "Harmonic oscillator",
                  "chain_obc": "Coupled chain (OBC)",
                  "chain_pbc": "Coupled chain (PBC)"}[sys_name], fontsize=12)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle(f"VMC accuracy vs model size  (N={N_PARTICLES} particles, {N_EPOCHS} epochs)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("error_vs_params.png", dpi=150, bbox_inches="tight")
plt.show()

### Error vs depth and width (heatmaps)

In [ ]:
n_systems = len(SYSTEMS)
n_archs   = len(ARCH_TYPES)

fig, axes = plt.subplots(n_systems, n_archs, figsize=(5 * n_archs, 4 * n_systems))

for row_idx, sys_name in enumerate(["HO", "chain_obc", "chain_pbc"]):
    for col_idx, arch_type in enumerate(ARCH_TYPES):
        ax  = axes[row_idx, col_idx]
        sub = df[(df["system"] == sys_name) & (df["arch"] == arch_type)]

        # Build heatmap: rows = depth, cols = width
        mat = np.full((len(HIDDEN_DEPTHS), len(HIDDEN_WIDTHS)), np.nan)
        for i, d in enumerate(HIDDEN_DEPTHS):
            for j, w in enumerate(HIDDEN_WIDTHS):
                row = sub[(sub["depth"] == d) & (sub["width"] == w)]
                if len(row) > 0:
                    mat[i, j] = float(row["rel_error"].values[0])

        im = ax.imshow(np.log10(mat + 1e-10), aspect="auto", cmap="viridis_r")
        ax.set_xticks(range(len(HIDDEN_WIDTHS)))
        ax.set_xticklabels(HIDDEN_WIDTHS)
        ax.set_yticks(range(len(HIDDEN_DEPTHS)))
        ax.set_yticklabels(HIDDEN_DEPTHS)
        ax.set_xlabel("Width")
        ax.set_ylabel("Depth")
        ax.set_title(f"{sys_name} – {ARCH_LABELS[arch_type]}", fontsize=9)
        plt.colorbar(im, ax=ax, label="log₁₀(rel. error)")

plt.suptitle("Relative energy error heatmap (depth × width)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("error_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

### Energy convergence curves (best run per system × architecture)

In [ ]:
fig, axes = plt.subplots(n_systems, n_archs, figsize=(5 * n_archs, 4 * n_systems),
                         sharex=True)

for row_idx, (sys_name, (_, E_exact)) in enumerate(SYSTEMS.items()):
    for col_idx, arch_type in enumerate(ARCH_TYPES):
        ax  = axes[row_idx, col_idx]

        # find best-performing (width, depth) for this system × arch
        sub = df[(df["system"] == sys_name) & (df["arch"] == arch_type)]
        if sub.empty:
            continue
        best_row = sub.loc[sub["error"].idxmin()]
        best_key = (sys_name, arch_type, int(best_row["width"]), int(best_row["depth"]))

        energies = results[best_key]["energies"]
        steps    = np.arange(len(energies))

        ax.plot(steps, energies, lw=0.8, color=ARCH_COLORS[arch_type], alpha=0.7)
        ax.axhline(E_exact, color="red", ls="--", lw=1.5,
                   label=f"E₀={E_exact:.3f}")
        ax.set_title(
            f"{sys_name} – {ARCH_LABELS[arch_type]}\n"
            f"best: w={int(best_row['width'])}, d={int(best_row['depth'])}",
            fontsize=8,
        )
        ax.set_ylabel("Energy")
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle("Energy convergence – best run per (system, architecture)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("convergence_best.png", dpi=150, bbox_inches="tight")
plt.show()

## Save results to disk

In [ ]:
import pickle, pathlib

out_dir = pathlib.Path("results")
out_dir.mkdir(exist_ok=True)

# Save DataFrame
df.to_csv(out_dir / "summary.csv", index=False)
print(f"Saved summary to {out_dir / 'summary.csv'}")

# Save full energy curves (no JAX arrays – already converted to np above)
curves = {str(k): v["energies"] for k, v in results.items()}
np.savez(out_dir / "energy_curves.npz", **{str(k): v for k, v in curves.items()})
print(f"Saved energy curves to {out_dir / 'energy_curves.npz'}")